# Python Tool Registry 구조 이해하기

이 노트북은 `BaseTool`, `AddTool`, `HelloTool`, `ToolRegistry` 구조를 단계별로 설명합니다.

핵심 구조는 다음과 같습니다.

```text
사용자 코드
   ↓
ToolRegistry.call()
   ↓
이름으로 Tool 검색
   ↓
Tool.__call__()
   ↓
Tool.run()
   ↓
실제 기능 실행
```


## 1. 전체 주요 기능

| 구성요소 | 역할 |
|---|---|
| `BaseTool` | 모든 Tool의 공통 형식 정의 |
| `schema()` | Tool의 이름, 설명, 파라미터 반환 |
| `run()` | 실제 기능 수행 |
| `__call__()` | 객체를 함수처럼 호출 가능하게 함 |
| `AddTool` | 두 수를 더하는 Tool |
| `HelloTool` | 인사말을 만드는 Tool |
| `ToolRegistry` | Tool 등록 및 관리 |
| `add()` | Tool 등록 |
| `list_tools()` | 등록된 Tool의 명세 조회 |
| `call()` | 이름으로 Tool 검색 후 실행 |
| `main()` | 전체 실행 흐름 테스트 |


## 2. BaseTool 만들기

`BaseTool`은 모든 Tool이 따라야 하는 부모 클래스입니다.

- `name`: 도구 이름
- `description`: 도구 설명
- `parameters`: 입력 파라미터 정보
- `schema()`: 도구 정보를 딕셔너리로 반환
- `run()`: 실제 기능 실행
- `__call__()`: 객체를 함수처럼 호출


In [ ]:
from typing import Any


class BaseTool:
    """모든 도구의 부모 클래스"""

    name: str = ""
    description: str = ""
    parameters: dict = {}

    def schema(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "parameters": self.parameters,
        }

    async def run(self, **kwargs) -> Any:
        raise NotImplementedError

    async def __call__(self, **kwargs) -> Any:
        return await self.run(**kwargs)


## 3. AddTool 만들기

`AddTool`은 `BaseTool`을 상속하고, `run()`만 구체적으로 구현합니다.

```text
AddTool 객체 호출
    ↓
__call__()
    ↓
run(a, b)
    ↓
a + b
```


In [ ]:
class AddTool(BaseTool):
    name = "add"
    description = "두 수를 더한다"
    parameters = {"a": "number", "b": "number"}

    async def run(self, a: float, b: float) -> float:
        return a + b


## 4. HelloTool 만들기

`who`를 입력하지 않으면 기본값으로 `"세상"`이 사용됩니다.


In [ ]:
class HelloTool(BaseTool):
    name = "hello"
    description = "인사말을 돌려준다"
    parameters = {"who": "string (선택)"}

    async def run(self, who: str = "세상") -> str:
        return f"안녕, {who}!"


## 5. ToolRegistry 만들기

`ToolRegistry`는 여러 Tool을 하나의 딕셔너리에 저장하고 관리합니다.

예를 들어 내부 상태는 다음과 같습니다.

```python
{
    "add": AddTool(),
    "hello": HelloTool()
}
```

`call()`이 가장 중요한 함수입니다.

```text
registry.call("add", {"a": 3, "b": 4})
                ↓
        _tools["add"] 검색
                ↓
            AddTool 객체
                ↓
         AddTool(a=3, b=4)
                ↓
            __call__()
                ↓
            run(3, 4)
                ↓
                7
```


In [ ]:
class ToolRegistry:
    """도구를 등록하고 이름으로 실행하는 관리자"""

    def __init__(self, name: str = "default") -> None:
        self.name = name
        self._tools: dict[str, BaseTool] = {}

    def add(self, tool: BaseTool) -> BaseTool:
        if tool.name in self._tools:
            raise ValueError(f"이미 있는 이름: {tool.name}")

        self._tools[tool.name] = tool
        return tool

    def list_tools(self) -> list[dict]:
        return [tool.schema() for tool in self._tools.values()]

    async def call(self, name: str, arguments: dict | None = None) -> Any:
        if name not in self._tools:
            raise KeyError(
                f"없는 도구: {name} (가능: {', '.join(self._tools)})"
            )

        return await self._tools[name](**(arguments or {}))


## 6. Tool 등록하기


In [ ]:
registry = ToolRegistry()

registry.add(AddTool())
registry.add(HelloTool())

print(registry._tools)


## 7. 등록된 Tool 명세 확인하기

`list_tools()`는 각 Tool의 `schema()`를 호출합니다.


In [ ]:
for spec in registry.list_tools():
    print(spec)


## 8. Tool 실행하기

Jupyter Notebook에서는 `await`를 직접 사용할 수 있습니다.


In [ ]:
result = await registry.call("add", {"a": 3, "b": 4})
print("add(3, 4) =", result)


In [ ]:
result = await registry.call("hello")
print("hello() =", result)


In [ ]:
result = await registry.call("hello", {"who": "민수"})
print("hello(민수) =", result)


## 9. 예외 처리 테스트

### 중복 Tool 등록


In [ ]:
try:
    registry.add(AddTool())
except ValueError as e:
    print("중복 등록 에러:", e)


### 없는 Tool 호출


In [ ]:
try:
    await registry.call("multiply")
except KeyError as e:
    print("없는 도구 에러:", e)


## 10. 전체 호출 프로세스

```text
사용자 / main
    │
    ▼
registry.call(name, arguments)
    │
    ▼
name 존재 여부 확인
    │
    ├── 없음 ──→ KeyError
    │
    ▼
_tools[name]에서 객체 가져오기
    │
    ▼
Tool 객체 호출
    │
    ▼
BaseTool.__call__()
    │
    ▼
self.run(**kwargs)
    │
    ├── AddTool.run()   → a + b
    │
    └── HelloTool.run() → 인사말 생성
    │
    ▼
결과 반환
```


## 11. 핵심 정리

이 구조의 핵심은 다음 4가지입니다.

1. `BaseTool`이 공통 규칙을 정의합니다.
2. 실제 Tool은 `run()`만 구현하면 됩니다.
3. `ToolRegistry`가 Tool을 이름으로 관리합니다.
4. `call()` → `__call__()` → `run()` 순서로 실제 기능이 실행됩니다.

이 패턴은 이후 MCP Tool, FastAPI Tool API, 함수 자동 등록 구조로 확장할 수 있습니다.
